# LANDFIRE Vegetation, Fuels, and Terrain — Exploratory Analysis

**Project:** `landfire-exploration`  
**Study area:** Central Colorado Mountain Landscape — Royal Gorge / Wet Mountains (~26 × 23 km)  
**LANDFIRE version:** LF2024 (version 240)  

---

## Scientific question

> *How do vegetation type, vegetation structure, and disturbance history map onto fuel model assignments across a heterogeneous mountain landscape — and how much of that mapping can a simple classifier recover from the vegetation layers alone?*

This notebook covers:
1. Study area setup and data acquisition
2. Loading and verifying alignment of LANDFIRE rasters
3. Landscape visualization (Fuel Vegetation Type, Cover, Height, FBFM40, disturbance)
4. Exploratory analysis: vegetation–fuel relationships
5. Educational classifier: predicting FBFM40 from vegetation variables
6. Adding terrain (elevation, slope, aspect)

**Pre-requisites:**  
Place downloaded LF2024 GeoTIFFs in `data/raw/` (see README for layer list).

## 1  Setup

In [ ]:
import sys
from pathlib import Path

# Locate project root (contains src/)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / 'src').exists() else _cwd.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError(
        f'Cannot find project root from {_cwd}. '
        'Launch Jupyter from the project root.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

from src import (
    PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, FIGURES_DIR, RESULTS_DIR,
    BBOX_WGS84, STUDY_AREA_NAME, LANDFIRE_VERSION, LANDFIRE_LAYERS,
)
from src.download_landfire import download_landfire, check_raw_data
from src.load_data import load_all_layers, build_dataset, FBFM40_LOOKUP, FBFM40_GROUP_COLORS
from src.align_rasters import verify_alignment, align, clip_to_bbox, build_aligned_dataset, save_processed
from src.terrain import load_or_download_dem, compute_terrain_layers, add_terrain_to_dataset
from src.analysis import (
    raster_to_dataframe, summarize_fuel_classes,
    plot_landscape_maps, plot_fuel_distributions,
    plot_fuel_evt_heatmap, plot_disturbance_effect,
)
from src.modeling import (
    prepare_features, train_fuel_classifier,
    plot_confusion_matrix, plot_feature_importance, plot_per_class_f1, save_metrics,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

print(f'Study area : {STUDY_AREA_NAME}')
print(f'BBOX WGS84 : {BBOX_WGS84}')
print(f'LANDFIRE   : LF{LANDFIRE_VERSION}')

## 2  Data Acquisition

The cell below will attempt to download LANDFIRE data automatically via the LFPS API.  
If the API is unavailable, it prints manual download instructions and the notebook continues, expecting files in `data/raw/`.

In [ ]:
# Check what's already in data/raw/
print('Checking raw data directory...')
status = check_raw_data(RAW_DIR)
missing = [k for k, v in status.items() if v is None]

if missing:
    print(f'\nMissing layers: {missing}')
    print('Attempting LFPS API download...')
    success = download_landfire(
        bbox_wgs84=BBOX_WGS84,
        output_dir=RAW_DIR,
    )
    if not success:
        print('\nDownload failed — place GeoTIFFs in data/raw/ and re-run.')
else:
    print('\nAll LANDFIRE layers present.')

## 3  Load and Align Raster Layers

Each layer is loaded as an `xr.DataArray`. We verify that all layers share the same CRS, resolution, and spatial extent before proceeding. Any mismatches are resolved by `align()`.

In [ ]:
print('Loading LANDFIRE layers from data/raw/...')
raw_layers = load_all_layers(RAW_DIR)

In [ ]:
# Verify alignment before combining into a Dataset
print('\nVerifying alignment:')
all_aligned = verify_alignment(raw_layers)

In [ ]:
# Layers downloaded from the LANDFIRE viewer are already clipped to your AOI,
# so we skip clip_to_bbox and go straight to alignment.
print('Aligning layers to common grid...')
aligned_layers = align(
    raw_layers,
    reference_key='evt',  # use EVT/FVT as the reference grid
    categorical_keys=['evt', 'fuel_model', 'disturbance'],
)

# Build the combined Dataset
ds = build_aligned_dataset(aligned_layers)
print(f'\nDataset:\n{ds}')

In [ ]:
# Save aligned layers to data/processed/ for fast reloading
print('Saving aligned layers to data/processed/...')
save_processed(aligned_layers, PROCESSED_DIR)

### Dataset overview

Quick statistics for each layer to confirm the data loaded correctly and values are in expected ranges.

In [ ]:
for var in ds.data_vars:
    arr = ds[var].values.ravel()
    arr = arr[~np.isnan(arr)]
    n_unique = len(np.unique(arr))
    print(f"  {var:<16}  n={len(arr):>7,}  unique={n_unique:>5}  "
          f"min={arr.min():.1f}  max={arr.max():.1f}  mean={arr.mean():.1f}")

## 4  Landscape Maps

Visualizing all layers side-by-side lets us compare how the same landscape is represented across different LANDFIRE products. Note that FBFM40 is shown by *fuel group* rather than individual model code, to keep the legend readable.

In [ ]:
fig = plot_landscape_maps(ds, figures_dir=FIGURES_DIR)
plt.show()

![Landscape maps: EVT, EVC, EVH, FBFM40, disturbance](../figures/landscape_maps.png)

**Landscape observations to look for:**

- Lower-montane areas (ponderosa pine, shrubland) should show Grass (GR), Shrub (SH), or
   Timber Understory (TU) fuel groups in FBFM40.
  - Where mixed conifer or denser forest appears in the Fuel Vegetation Type layer, expect
   Timber Litter (TL) or Timber Understory (TU) fuel groups.                              
  - The Wet Mountains terrain will likely drive visible aspect-related patterns —
  south-facing slopes tend toward drier shrub/grass fuel classes while north-facing slopes
   retain more forest litter models.
  - Fuel Vegetation Type and FBFM40 are correlated but not identical — fuel models also   
  encode structure (cover, height) and fire-behavior assumptions, so adjacent cells with  
  similar vegetation can map to different fuel classes.

## 5  Vegetation–Fuel Exploratory Analysis

Convert the aligned rasters to a flat DataFrame for cell-level statistics.

In [ ]:
df = raster_to_dataframe(ds, max_pixels=300_000)
df.head()

### 5.1  Fuel class summary

In [ ]:
summary = summarize_fuel_classes(df)
print(f'FBFM40 classes present in study area: {len(summary)}')
summary.round(2)

### 5.2  EVC and EVH distributions by fuel group

Do fuel model groups occupy distinct vegetation-cover and height ranges? Timber Litter models (TL) should occupy cells with significant tree canopy height; Grass models (GR) should have low height and moderate cover.

In [ ]:
fig = plot_fuel_distributions(df, figures_dir=FIGURES_DIR)
plt.show()

![EVC and EVH distributions by fuel group](../figures/fuel_distributions.png)

### 5.3  EVT composition within fuel model groups

A heatmap showing which EVT classes occupy each fuel model group. Where one fuel group contains many EVT classes, the mapping is ecologically heterogeneous and a simple spectral/structural classifier will struggle to distinguish it from neighboring groups.

In [ ]:
fig = plot_fuel_evt_heatmap(df, top_n_evt=15, figures_dir=FIGURES_DIR)
plt.show()

![EVT composition within fuel model groups](../figures/fuel_evt_heatmap.png)

### 5.4  Effect of disturbance on fuel model distribution

Disturbance (fire, harvest, drought) can shift cells from one fuel group to another. The Cameron Peak Fire area provides a visible natural experiment: cells within the fire perimeter should show a different fuel group composition than adjacent undisturbed cells.

In [ ]:
# What fraction of cells have recent disturbance recorded?
dist_frac = (df['disturbance'] > 0).mean() if 'disturbance' in df.columns else None
print(f'Fraction of study-area cells with recorded disturbance: {dist_frac:.1%}')

fig = plot_disturbance_effect(df, figures_dir=FIGURES_DIR)
plt.show()

![Fuel group distribution: disturbed vs. undisturbed cells](../figures/disturbance_effect.png)

## 6  Predictive Experiment: FBFM40 from Vegetation Variables

### Scientific question
> How much of the mapped fuel class can be recovered from EVT, EVC, EVH, and disturbance presence alone — and where does that simplified relationship break down?

### What this is NOT
This classifier does **not** reproduce LANDFIRE's fuel-assignment process. LANDFIRE's fuel model assignments incorporate expert rules, ecological context, and knowledge accumulated over decades of field validation. A Random Forest trained on the same input variables will partially recover those relationships but cannot replicate the full decision logic.

Errors are informative: they identify which fuel classes require contextual information (species composition, forest age, fuel load measurements) that is not captured in these aggregate vegetation layers.

In [ ]:
X, y, feature_names, encoder = prepare_features(
    df,
    target_col='fuel_model',
    feature_cols=['evt', 'evc', 'evh', 'disturbance'],
    min_class_size=30,
)
print(f'\nFeature matrix: {X.shape}')
print(f'Classes: {[str(c) for c in encoder.classes_]}')

In [ ]:
clf, metrics = train_fuel_classifier(
    X, y,
    n_estimators=200,
    test_size=0.25,
    seed=42,
)
print('\nNote: random split — nearby cells in train/test share spatial context.')
print('This likely inflates accuracy vs. a spatially blocked evaluation.')

In [ ]:
y_test = np.array(metrics['y_test'])
y_pred = np.array(metrics['y_pred'])

fig = plot_confusion_matrix(y_test, y_pred, encoder, figures_dir=FIGURES_DIR)
plt.show()

![FBFM40 confusion matrix](../figures/fuel_confusion_matrix.png)

In [ ]:
fig = plot_feature_importance(clf, feature_names, top_n=20, figures_dir=FIGURES_DIR)
plt.show()

![Feature importance](../figures/feature_importance.png)

In [ ]:
import matplotlib.patches as mpatches
from src.load_data import FBFM40_GROUP_COLORS, fbfm40_group, fbfm40_name
from sklearn.metrics import classification_report                                        
   
classes = encoder.classes_                                                               
report = classification_report(
      y_test, y_pred,
      labels=range(len(classes)),
      target_names=[fbfm40_name(int(c)) for c in classes],
      output_dict=True,
      zero_division=0,
  )
names  = [fbfm40_name(int(c)) for c in classes]
f1s    = [report[n]["f1-score"] for n in names]                                          
colors = [FBFM40_GROUP_COLORS.get(fbfm40_group(int(c)), "#888888") for c in classes]
order  = np.argsort(f1s)[::-1]                                                           
                  
fig, ax = plt.subplots(figsize=(12, max(4, len(names) * 0.38)))                          
ax.barh(range(len(names)), [f1s[i] for i in order],
          color=[colors[i] for i in order], edgecolor="white")                             
ax.set_yticks(range(len(names)))                                                         
ax.set_yticklabels([names[i] for i in order], fontsize=8)
ax.set_xlabel("F1 score")                                                                
ax.set_xlim(0, 1.05)                                                                     
ax.set_title("Per-class F1: FBFM40 Prediction from FVT + FVC + FVH + Disturbance",
               fontsize=11, fontweight="bold")                                             
ax.xaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)                                                                   
patches = [mpatches.Patch(color=c, label=g)
             for g, c in FBFM40_GROUP_COLORS.items()                                       
             if any(fbfm40_group(int(cls)) == g for cls in classes)]
ax.legend(handles=patches, loc="lower right", fontsize=7, framealpha=0.8)                
fig.tight_layout()
fig.savefig(FIGURES_DIR / "per_class_f1.png", dpi=150, bbox_inches="tight")              
plt.show()      
                                                                                           
save_metrics(metrics, RESULTS_DIR)

![Per-class F1: FBFM40 prediction from vegetation variables](../figures/per_class_f1.png)

### 6.1  Interpreting the errors

Classes with low F1 scores typically represent fuel models that:
- Span multiple EVT classes (ecologically heterogeneous)
- Are distinguished from adjacent classes by fuel load details (not just cover/height)
- Reflect disturbance history that changed fuel structure without changing EVT
- Require sub-patch structural information (snag density, duff depth, fuel continuity)

For example, adjacent Timber Litter models (TL1 vs. TL2) may have nearly identical EVT, EVC, and EVH — they differ in compaction and load characteristics calibrated from field measurements, which are not captured by these remote-sensing layers.

## 7  Terrain Variables

Topography is a quasi-static conditioning factor in the fire-behavior environment. Here we derive elevation, slope, and aspect from the USGS 3DEP DEM.

**Fire-behavior context:**
- **Slope** directly accelerates fire spread (Rothermel model). Steeper slopes increase the effective rate of spread.
- **Aspect** controls insolation, drying rate, and fuel moisture. South-facing slopes in the Northern Hemisphere receive more direct sun, produce drier fuels, and are more prone to fire activity.
- **Elevation** drives vegetation and fuel type through temperature lapse rate and precipitation gradients.

In [ ]:
dem = load_or_download_dem(BBOX_WGS84, resolution=30, processed_dir=PROCESSED_DIR)
print(f'DEM shape: {dem.shape}  CRS: {dem.rio.crs}')
print(f'Elevation range: {float(dem.min()):.0f} – {float(dem.max()):.0f} m')

In [ ]:
terrain_layers = compute_terrain_layers(dem, output_dir=PROCESSED_DIR)
ds_terrain = add_terrain_to_dataset(ds, terrain_layers)

In [ ]:
# Map terrain variables
terrain_ds = xr.Dataset({k: terrain_layers[k] for k in terrain_layers})
fig = plot_landscape_maps(terrain_ds, figures_dir=FIGURES_DIR)
plt.show()

In [ ]:
# Add terrain to DataFrame and look at fuel-class vs. slope/aspect
df_terrain = raster_to_dataframe(ds_terrain, max_pixels=300_000)

if 'slope' in df_terrain.columns and 'fuel_model' in df_terrain.columns:
    from src.load_data import fbfm40_group
    df_terrain['fuel_group'] = df_terrain['fuel_model'].apply(
        lambda c: fbfm40_group(int(c)) if not np.isnan(c) else 'Unknown'
    )
    slope_by_group = (
        df_terrain.dropna(subset=['slope', 'fuel_group'])
        .groupby('fuel_group')['slope']
        .median()
        .sort_values(ascending=False)
    )
    print('Median slope by fuel group:')
    print(slope_by_group.round(1).to_string())

## 8  Summary

| Step | Finding |
|---|---|
| Data loading | LANDFIRE LF2022 layers loaded and aligned to UTM Zone 13N, 30 m grid |
| Landscape visualization | EVT diversity, FBFM40 spatial structure, and disturbance visible in maps |
| Vegetation–fuel EDA | TL and TU groups occupy higher canopy; GR/GS groups are structurally low |
| Disturbance effect | Disturbed cells show different fuel group composition (SB, GR increase) |
| Classifier | RF recovers ~XX% of fuel-model assignments; structurally similar fuel models confused |
| Terrain | Slope and aspect vary substantially across the study area; fuel groups differ by terrain |

*(Fill in accuracy from the cell above after running)*

**Proceed to `02_weather_integration.ipynb`** to add ERA5 fire-weather data and build the integrated fire-behavior environment visualization.